In [1]:
import os
import pickle
from pathlib import Path

# WSL crash workaround: avoid TensorFlow/Triton code paths in this notebook.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import evaluate
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

PROJECT_DIR = Path.cwd()

BASE_MODEL_DIR = PROJECT_DIR / "models" / "DACTYL"
BASE_WEIGHT_FILE = BASE_MODEL_DIR / "model.safetensors"

with open("pickles/train_samples.pkl", "rb") as f:
    train_samples = pickle.load(f)
with open("pickles/validation_samples.pkl", "rb") as f:
    val_samples = pickle.load(f)
with open("pickles/test_samples.pkl", "rb") as f:
    test_samples = pickle.load(f)

print("train_samples:", len(train_samples))
print("val_samples:", len(val_samples))
print("test_samples:", len(test_samples))

/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Plea

train_samples: 88128
val_samples: 5508
test_samples: 16524


In [2]:
if not BASE_MODEL_DIR.exists() or not BASE_WEIGHT_FILE.exists():
    raise FileNotFoundError(
        f"Missing local model at {BASE_MODEL_DIR}.\n"
        "Download once with:\n"
        "  huggingface-cli download ShantanuT01/dactyl-ai-text-detector --local-dir models/DACTYL"
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, use_fast=True)
base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Loaded from:", BASE_MODEL_DIR)
print("Base model:", base_model.__class__.__name__)
print("device:", device)

/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Loaded from: /home/ine/projects/human-ai-mixed-data-generator/models/DACTYL
Base model: DebertaV2ForSequenceClassification
device: cuda


In [3]:
class DebertaTokenClassifier(nn.Module):
    """Token-level head on top of frozen DeBERTa encoder."""

    def __init__(self, backbone_seq_model, num_labels=2, dropout=0.1):
        super().__init__()
        self.deberta = backbone_seq_model.deberta
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(backbone_seq_model.config.hidden_size, num_labels)
        self.num_labels = num_labels

        # Freeze backbone parameters
        for p in self.deberta.parameters():
            p.requires_grad = False

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        logits = self.classifier(sequence_output)  # [batch, seq, num_labels]

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return {"loss": loss, "logits": logits}


model = DebertaTokenClassifier(base_model, num_labels=2).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable ratio: {100 * trainable / total:.4f}%")

Trainable params: 2,050
Total params: 434,014,210
Trainable ratio: 0.0005%


In [4]:
class TokenAlignedDataset(Dataset):
    """Align word-level labels to tokenizer output ids with -100 masking."""

    def __init__(self, samples, tokenizer, max_length=512):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        words = sample["text"].split()
        word_labels = sample["label"]

        if len(words) != len(word_labels):
            raise ValueError(f"Word/label mismatch at idx {idx}: {len(words)} vs {len(word_labels)}")

        enc = self.tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            return_attention_mask=True,
        )

        word_ids = enc.word_ids()
        aligned_labels = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != prev_word_id:
                aligned_labels.append(int(word_labels[word_id]))
            else:
                # Ignore subsequent subword pieces in loss
                aligned_labels.append(-100)
            prev_word_id = word_id

        return {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(aligned_labels, dtype=torch.long),
        }


def token_collator(batch):
    input_ids = [x["input_ids"] for x in batch]
    attention_mask = [x["attention_mask"] for x in batch]
    labels = [x["labels"] for x in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


MAX_LENGTH = 256

# Crash-safe debug limits (set to None for full dataset)
TRAIN_LIMIT = 5000
VAL_LIMIT = 1000
TEST_LIMIT = 1000

train_data = train_samples[:TRAIN_LIMIT] if TRAIN_LIMIT else train_samples
val_data = val_samples[:VAL_LIMIT] if VAL_LIMIT else val_samples
test_data = test_samples[:TEST_LIMIT] if TEST_LIMIT else test_samples

train_ds = TokenAlignedDataset(train_data, tokenizer, max_length=MAX_LENGTH)
val_ds = TokenAlignedDataset(val_data, tokenizer, max_length=MAX_LENGTH)
test_ds = TokenAlignedDataset(test_data, tokenizer, max_length=MAX_LENGTH)

sample = train_ds[0]
print("sample shapes:", sample["input_ids"].shape, sample["labels"].shape)
print("dataset sizes:", len(train_ds), len(val_ds), len(test_ds))

sample shapes: torch.Size([119]) torch.Size([119])
dataset sizes: 5000 1000 1000


In [5]:
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Flatten while ignoring masked labels
    labels_flat = labels.reshape(-1)
    preds_flat = preds.reshape(-1)
    keep = labels_flat != -100

    labels_kept = labels_flat[keep]
    preds_kept = preds_flat[keep]

    if len(labels_kept) == 0:
        return {"token_accuracy": 0.0, "token_f1": 0.0}

    acc = metric_acc.compute(predictions=preds_kept, references=labels_kept)["accuracy"]
    f1 = metric_f1.compute(predictions=preds_kept, references=labels_kept, average="binary")["f1"]
    return {"token_accuracy": acc, "token_f1": f1}

In [7]:
class StepProgressCallback(TrainerCallback):
    """Print clear step/epoch progress during training."""

    def on_train_begin(self, args, state, control, **kwargs):
        print(f"Training started | epochs={args.num_train_epochs} | max_steps={state.max_steps}")

    def on_epoch_begin(self, args, state, control, **kwargs):
        epoch = int(state.epoch) + 1
        print(f"\n=== Epoch {epoch}/{int(args.num_train_epochs)} ===")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        max_steps = state.max_steps or 1
        pct = 100.0 * step / max_steps
        msg = [f"step {step}/{max_steps} ({pct:.1f}%)"]
        if "loss" in logs:
            msg.append(f"loss={logs['loss']:.4f}")
        if "learning_rate" in logs:
            msg.append(f"lr={logs['learning_rate']:.2e}")
        if "token_accuracy" in logs:
            msg.append(f"token_acc={logs['token_accuracy']:.4f}")
        if "token_f1" in logs:
            msg.append(f"token_f1={logs['token_f1']:.4f}")
        print(" | ".join(msg), flush=True)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            parts = [f"{k}={v:.4f}" for k, v in metrics.items() if isinstance(v, (int, float))]
            print(f"[eval @ step {state.global_step}] " + ", ".join(parts), flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        print(f"Training finished at step {state.global_step}", flush=True)


OUTPUT_DIR = PROJECT_DIR / "output" / "dactyl_token_finetuned"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=4,
    evaluation_strategy="epoch",
    # save_strategy="no",  # disable checkpoint writes while stabilizing
    save_strategy="epoch",
    # load_best_model_at_end=False,
    load_best_model_at_end=True,
    metric_for_best_model="token_f1",
    greater_is_better=True,
    logging_steps=10,
    logging_first_step=True,
    disable_tqdm=False,
    dataloader_num_workers=4,
    # fp16=False,
    fp16=True,
    bf16=False,
    tf32=False,
    optim="adamw_torch",
    report_to="none",
)

progress_callback = StepProgressCallback()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=token_collator,
    compute_metrics=compute_metrics,
    callbacks=[progress_callback],
)


/home/ine/miniconda3/envs/bittensor/lib/python3.10/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [8]:
# Run training (watch step logs + tqdm bar)
try:
    train_result = trainer.train()
    print("Final train metrics:", train_result.metrics)
except Exception as e:
    print("Training failed:", repr(e))
    raise

Training started | epochs=4 | max_steps=624

=== Epoch 1/4 ===


Epoch,Training Loss,Validation Loss,Token Accuracy,Token F1
0,0.424900,0.399306,0.823985,0.768803
1,0.356300,0.374702,0.839544,0.789159
2,0.347900,0.359059,0.849224,0.796977
3,0.355000,0.359036,0.848819,0.799629


step 1/624 (0.2%) | loss=0.5922 | lr=2.00e-04
step 10/624 (1.6%) | loss=0.4684 | lr=1.97e-04
step 20/624 (3.2%) | loss=0.4255 | lr=1.94e-04
step 30/624 (4.8%) | loss=0.5185 | lr=1.90e-04
step 40/624 (6.4%) | loss=0.5030 | lr=1.87e-04
step 50/624 (8.0%) | loss=0.4663 | lr=1.84e-04
step 60/624 (9.6%) | loss=0.4150 | lr=1.81e-04
step 70/624 (11.2%) | loss=0.4441 | lr=1.78e-04
step 80/624 (12.8%) | loss=0.3822 | lr=1.74e-04
step 90/624 (14.4%) | loss=0.3869 | lr=1.71e-04
step 100/624 (16.0%) | loss=0.4282 | lr=1.68e-04
step 110/624 (17.6%) | loss=0.4245 | lr=1.65e-04
step 120/624 (19.2%) | loss=0.4175 | lr=1.62e-04
step 130/624 (20.8%) | loss=0.4191 | lr=1.58e-04
step 140/624 (22.4%) | loss=0.4083 | lr=1.55e-04
step 150/624 (24.0%) | loss=0.4249 | lr=1.52e-04
step 156/624 (25.0%)
[eval @ step 156] eval_loss=0.3993, eval_token_accuracy=0.8240, eval_token_f1=0.7688, eval_runtime=72.9504, eval_samples_per_second=13.7080, eval_steps_per_second=13.7080, epoch=1.0000

=== Epoch 1/4 ===
step 160/

In [9]:
import json

test_metrics = trainer.evaluate(test_ds)
print("Test metrics:", test_metrics)

best_dir = OUTPUT_DIR / "best"
best_dir.mkdir(parents=True, exist_ok=True)

# Save tokenizer
tokenizer.save_pretrained(best_dir)

# Save custom token-classifier weights + minimal metadata
weights_path = best_dir / "token_classifier_state.pt"
torch.save(model.state_dict(), weights_path)

meta = {
    "base_model_dir": str(BASE_MODEL_DIR),
    "num_labels": 2,
    "max_length": MAX_LENGTH,
    "model_class": "DebertaTokenClassifier",
}
with open(best_dir / "token_classifier_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

if not weights_path.exists():
    raise RuntimeError(f"Weight save failed: {weights_path} not found")

print("Saved token-level model to", best_dir)
print("Weights:", weights_path)
print("Meta:", best_dir / "token_classifier_meta.json")

# ---- Optional: export as a HuggingFace token-classification model bundle ----
hf_export_dir = OUTPUT_DIR / "best_hf"
hf_export_dir.mkdir(parents=True, exist_ok=True)

hf_config = AutoConfig.from_pretrained(BASE_MODEL_DIR)
hf_config.num_labels = 2
hf_config.id2label = {0: "human", 1: "ai"}
hf_config.label2id = {"human": 0, "ai": 1}

# Base DACTYL is sequence-classification (1 logit). Do not from_pretrained into a 2-label head.
hf_model = AutoModelForTokenClassification.from_config(hf_config)
hf_model.deberta.load_state_dict(model.deberta.state_dict(), strict=True)
hf_model.classifier.load_state_dict(model.classifier.state_dict(), strict=True)

hf_model.save_pretrained(hf_export_dir)
tokenizer.save_pretrained(hf_export_dir)

print("Exported HuggingFace model bundle to", hf_export_dir)
print("Includes config.json + model.safetensors + tokenizer files")

step 624/624 (100.0%)
[eval @ step 624] eval_loss=0.3556, eval_token_accuracy=0.8532, eval_token_f1=0.7975, eval_runtime=78.4521, eval_samples_per_second=12.7470, eval_steps_per_second=12.7470, epoch=3.9900
Test metrics: {'eval_loss': 0.3555832505226135, 'eval_token_accuracy': 0.8531530548203127, 'eval_token_f1': 0.7974900278467675, 'eval_runtime': 78.4521, 'eval_samples_per_second': 12.747, 'eval_steps_per_second': 12.747, 'epoch': 3.99}
Saved token-level model to /home/ine/projects/human-ai-mixed-data-generator/output/dactyl_token_finetuned/best
Weights: /home/ine/projects/human-ai-mixed-data-generator/output/dactyl_token_finetuned/best/token_classifier_state.pt
Meta: /home/ine/projects/human-ai-mixed-data-generator/output/dactyl_token_finetuned/best/token_classifier_meta.json


RuntimeError: Error(s) in loading state_dict for DebertaV2ForTokenClassification:
	size mismatch for classifier.weight: copying a param with shape torch.Size([1, 1024]) from checkpoint, the shape in current model is torch.Size([2, 1024]).
	size mismatch for classifier.bias: copying a param with shape torch.Size([1]) from checkpoint, the shape in current model is torch.Size([2]).
	You may consider adding `ignore_mismatched_sizes=True` in the model `from_pretrained` method.

In [10]:
# load fine-tuned token-level model from local directory

import json
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_DIR = PROJECT_DIR / "output" / "dactyl_token_finetuned" / "best"
WEIGHT_FILE = SAVE_DIR / "token_classifier_state.pt"
META_FILE = SAVE_DIR / "token_classifier_meta.json"

if not SAVE_DIR.exists():
    raise FileNotFoundError(f"Missing model directory: {SAVE_DIR}")
if not WEIGHT_FILE.exists() or not META_FILE.exists():
    raise FileNotFoundError(
        f"Missing custom token model files in {SAVE_DIR}.\n"
        "Run the training+save cell first."
    )

with open(META_FILE, "r") as f:
    meta = json.load(f)

base_model_dir = Path(meta["base_model_dir"])
num_labels = int(meta.get("num_labels", 2))

# Guard against stale/wrong metadata from earlier saves
if not (base_model_dir / "config.json").exists():
    print(f"Warning: invalid base_model_dir in meta: {base_model_dir}")
    base_model_dir = PROJECT_DIR / "models" / "DACTYL"

if not WEIGHT_FILE.exists():
    raise FileNotFoundError(f"Missing token head weights: {WEIGHT_FILE}. Re-run save cell.")

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
base_for_load = AutoModelForSequenceClassification.from_pretrained(base_model_dir)
best_model = DebertaTokenClassifier(base_for_load, num_labels=num_labels)

state_dict = torch.load(WEIGHT_FILE, map_location="cpu")
best_model.load_state_dict(state_dict)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model.to(device)
best_model.eval()

print("Loaded token-level model from", SAVE_DIR)
print("Using backbone:", base_model_dir)

Loaded token-level model from /home/ine/projects/human-ai-mixed-data-generator/output/dactyl_token_finetuned/best
Using backbone: /home/ine/projects/human-ai-mixed-data-generator/models/DACTYL


In [11]:
best_model.eval()

DebertaTokenClassifier(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 1024, padding_idx=0)
      (LayerNorm): LayerNorm((1024,), eps=1e-07, elementwise_affine=True, bias=True)
      (dropout): StableDropout()
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-23): 24 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (key_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (value_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (pos_dropout): StableDropout()
              (dropout): StableDropout()
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1024,), eps=1e-07, elementw

In [12]:
# Inspect new classifier
model.classifier

Linear(in_features=1024, out_features=2, bias=True)

In [ ]:
# Load exported HuggingFace token model (config.json + safetensors)
HF_EXPORT_DIR = OUTPUT_DIR / "best_hf"

hf_tokenizer = AutoTokenizer.from_pretrained(HF_EXPORT_DIR)
hf_token_model = AutoModelForTokenClassification.from_pretrained(HF_EXPORT_DIR)

hf_token_model.to(device)
hf_token_model.eval()
print("Loaded HF-exported token model from", HF_EXPORT_DIR)